<a href="https://colab.research.google.com/github/mkvkanpur/hpc/blob/main/Nple_3d_warp_jax.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
try:
    import warp as wp
    print(f"Warp version {wp.__version__} is ready!")
except ImportError:
    print("Warp not found. Installing...")
    !pip install warp-lang
    import warp as wp

wp.init()
device = "cuda"


Warp not found. Installing...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 MB 8.3 MB/s eta 0:00:00
Warp 1.11.1 initialized:
   CUDA Toolkit 12.9, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "Tesla T4" (15 GiB, sm_75, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.11.1


In [1]:
import jax
import jax.numpy as jnp

# Enable XLA GPU backend if available
jax.config.update('jax_platform_name', 'gpu')

# Simulation parameters
N_PARTICLES = 100
BOX_SIZE = 10.0
DT = 0.01  # Time step
NUM_STEPS = 1000

# Initialize particles: random positions and velocities
key = jax.random.PRNGKey(0)
key, subkey_pos, subkey_vel = jax.random.split(key, 3)

positions = jax.random.uniform(subkey_pos, (N_PARTICLES, 2), minval=0.0, maxval=BOX_SIZE)
velocities = jax.random.uniform(subkey_vel, (N_PARTICLES, 2), minval=-1.0, maxval=1.0)

@jax.jit
def simulate_step(pos, vel):
    # Update positions
    new_pos = pos + vel * DT

    # Handle boundary conditions (bounce off walls)
    # Check if particles hit the boundary
    hit_x_min = new_pos[:, 0] < 0.0
    hit_x_max = new_pos[:, 0] > BOX_SIZE
    hit_y_min = new_pos[:, 1] < 0.0
    hit_y_max = new_pos[:, 1] > BOX_SIZE

    # Reverse velocity component if hit
    new_vel_x = jnp.where(hit_x_min | hit_x_max, -vel[:, 0], vel[:, 0])
    new_vel_y = jnp.where(hit_y_min | hit_y_max, -vel[:, 1], vel[:, 1])
    new_vel = jnp.stack([new_vel_x, new_vel_y], axis=-1)

    # Correct positions to be within bounds
    new_pos = jnp.clip(new_pos, 0.0, BOX_SIZE)

    return new_pos, new_vel

# Run simulation
current_positions = positions
current_velocities = velocities

for _ in range(NUM_STEPS):
    current_positions, current_velocities = simulate_step(current_positions, current_velocities)

print(f"Final positions after {NUM_STEPS} steps:")
print(current_positions[:5]) # Print first 5 particles


Final positions after 1000 steps:
[[8.121947   8.454809  ]
 [2.635192   1.9761596 ]
 [5.197877   1.3435638 ]
 [5.5983963  0.68119025]
 [2.7704005  1.1981258 ]]


This simulation uses JAX to perform the calculations on the GPU (VRAM). It initializes `N_PARTICLES` particles with random positions and velocities within a `BOX_SIZE` x `BOX_SIZE` 2D area. In each time step (`DT`), the particles' positions are updated, and if they hit the boundary, their respective velocity component is reversed, simulating a bounce. The `@jax.jit` decorator compiles the `simulate_step` function for efficient execution on the GPU.

# WARP (VVRAM)

In [6]:
import numpy as np
import warp as wp
import time

wp.init()

@wp.kernel
def nbody_kernel(x: wp.array(dtype=wp.vec3),
                 a: wp.array(dtype=wp.vec3),
                 n: int,
                 G: float,
                 eps2: float):

    # Each thread handles exactly ONE particle i
    i = wp.tid()

    # Boundary check for non-power-of-two N
    if i >= n:
        return

    # Cache particle i's position in a register to reduce VRAM reads
    pos_i = x[i]
    accel_i = wp.vec3(0.0, 0.0, 0.0)

    # Loop over all other particles j in VRAM
    for j in range(n):
        if i == j:
            continue

        # Physics: r_ij = x[j] - x[i]
        r_ij = x[j] - pos_i

        # Softened distance squared
        dist_sq = wp.length_sq(r_ij) + eps2

        # High-performance 1/dist^3 calculation
        # rsqrt is much faster than 1/sqrt on NVIDIA GPUs
        r_sq = wp.length_sq(r_ij)
        inv_dist_cube = 1.0/wp.sqrt(r_sq) * (1.0 / r_sq)

        accel_i += r_ij * inv_dist_cube

    # Apply G and store back to VRAM
    a[i] = accel_i * G

def main():
    # Setup parameters
    N = 10000
    G = 6.674e-11
    epsilon = 0.1
    eps2 = epsilon * epsilon
    device = "cuda"

    # 1. Initialize data (Random positions)
    x_np = np.random.rand(N, 3).astype(np.float32)
    x_wp = wp.array(x_np, dtype=wp.vec3, device=device)
    a_wp = wp.zeros(N, dtype=wp.vec3, device=device)

    print(f"Starting simulation for N={N} on {wp.get_device()}...")

    # 2. Launch Kernel: One thread per particle
    start_time = time.time()

    wp.launch(
        kernel=nbody_kernel,
        dim=N,
        inputs=[x_wp, a_wp, N, G, eps2],
        device=device
    )

    wp.synchronize()
    end_time = time.time()

    # 3. Output results
    accel_out = a_wp.numpy()
    print("-" * 30)
    print(f"Computation Time: {(end_time - start_time)*1000:.2f} ms")
    print(f"Sample Acceleration [0]: {accel_out[0]}")
    print("-" * 30)

if __name__ == "__main__":
    main()

Starting simulation for N=10000 on cuda:0...
Module __main__ d9ac97e load on device 'cuda:0' took 993.97 ms  (compiled)
------------------------------
Computation Time: 1001.99 ms
Sample Acceleration [0]: [ 9.954705e-07 -6.513568e-07 -5.571707e-07]
------------------------------


# Shared mem WARP

In [8]:
import numpy as np
import warp as wp
import time

wp.init()

# The size of the chunk we process at once. 128 is a safe bet for most GPUs.
TILE_SIZE = 128

@wp.kernel
def nbody_tiled_kernel(x: wp.array(dtype=wp.vec3),
                       a: wp.array(dtype=wp.vec3),
                       n: int,
                       G: float,
                       eps2: float):

    # Map thread to particle i
    i = wp.tid()
    if i >= n:
        return

    # Cache target particle in a register (High Speed)
    pos_i = x[i]
    accel_i = wp.vec3(0.0, 0.0, 0.0)

    # TILING LOGIC:
    # Instead of one giant loop, we break 'j' into chunks (Tiles).
    # This structure is what the 'warp.tile' API automates.
    num_tiles = (n + TILE_SIZE - 1) // TILE_SIZE

    for t in range(num_tiles):
        # Determine the bounds of the current tile
        j_start = t * TILE_SIZE
        j_end = wp.min(j_start + TILE_SIZE, n)

        # Inner loop over the current tile
        for j in range(j_start, j_end):
            if i == j:
                continue

            r_ij = x[j] - pos_i
            dist_sq = wp.length_sq(r_ij) + eps2

            inv_dist_cube = 1.0/wp.sqrt(dist_sq) * (1.0 / dist_sq)

            accel_i += r_ij * inv_dist_cube

    # Store final acceleration
    a[i] = accel_i * G

def main():
    N = 10000
    G = 6.674e-11
    eps2 = 0.01
    device = "cuda"

    # Initialize data
    x_np = np.random.rand(N, 3).astype(np.float32)
    x_wp = wp.array(x_np, dtype=wp.vec3, device=device)
    a_wp = wp.zeros(N, dtype=wp.vec3, device=device)

    print(f"Launching TILED simulation for N={N} on {wp.get_device()}...")

    start_time = time.time()

    # Launch: Every particle i gets one thread.
    wp.launch(
        kernel=nbody_tiled_kernel,
        dim=N,
        inputs=[x_wp, a_wp, N, G, eps2],
        device=device
    )

    wp.synchronize()
    print(f"Tiled Computation Time: {(time.time() - start_time)*1000:.2f} ms")

if __name__ == "__main__":
    main()

Launching TILED simulation for N=10000 on cuda:0...
Module __main__ 24163d2 load on device 'cuda:0' took 567.76 ms  (compiled)
Tiled Computation Time: 576.65 ms


# FUTURE code with 1.2.1

In [ ]:
import numpy as np
import warp as wp
import warp.tile as wt # The 1.2.1 high-level Tile API
import time

wp.init()

# Use 128 or 256 for optimal hardware occupancy on RTX 6000
TILE_SIZE = 128

@wp.kernel
def nbody_tile_api_kernel(x: wp.array(dtype=wp.vec3),
                          a: wp.array(dtype=wp.vec3),
                          n: int,
                          G: float,
                          eps2: float):

    # 1. Thread index maps to target particle 'i'
    i = wp.tid()
    if i >= n:
        return

    # Cache target particle position in a register
    pos_i = x[i]
    accel_i = wp.vec3(0.0)

    # 2. Iterate through source particles 'j' using the Tile API
    # wt.tiles() automatically handles index math and boundary conditions
    for x_tile in wt.tiles(x, shape=(TILE_SIZE,)):

        # Load the current tile of particle positions into Shared Memory
        pos_j_tile = wt.tile_load(x_tile)

        # 3. Compute interactions for the entire tile at once
        # Displacement: subtract target pos_i from every particle in the tile
        r_ij_tile = pos_j_tile - pos_i

        # Squared distance for the whole tile
        dist_sq_tile = wt.length_sq(r_ij_tile) + eps2

        # Physics: 1 / (dist^3)
        inv_dist_cube_tile = wt.rsqrt(dist_sq_tile) / dist_sq_tile

        # Accumulation: Sum the tile results into our local register
        accel_i += wt.sum(r_ij_tile * inv_dist_cube_tile)

    # 4. Final force application
    a[i] = accel_i * G

# --- Main Execution Script ---
def main():
    # Parameters for class demonstration
    N = 8192  # Multiple of TILE_SIZE for simplicity
    G = 1.0   # Normalized G for classroom demo
    eps2 = 0.001
    device = "cuda"

    # Initialize data on CPU, then move to GPU
    x_np = np.random.rand(N, 3).astype(np.float32)
    x_wp = wp.array(x_np, dtype=wp.vec3, device=device)
    a_wp = wp.zeros(N, dtype=wp.vec3, device=device)

    print(f"Running Warp 1.2.1 Tile API Solver...")
    print(f"Target Hardware: {wp.get_device()}")

    start = time.time()

    # Launch: One thread per target particle
    wp.launch(
        kernel=nbody_tile_api_kernel,
        dim=N,
        inputs=[x_wp, a_wp, N, G, eps2],
        device=device
    )

    wp.synchronize()
    duration = (time.time() - start) * 1000

    print("-" * 30)
    print(f"Particles: {N}")
    print(f"Execution Time: {duration:.2f} ms")
    print(f"Sample Acceleration [0]: {a_wp.numpy()[0]}")
    print("-" * 30)

if __name__ == "__main__":
    main()